# VIX Deep Learning Pipeline — Version Finale Consolidée
## Toutes les architectures · Features TS complètes · Modules V2 · Anti-leakage · PreScaledDataset

**Sources combinées :** V1 Advanced + Enriched (features robustes) + V2 Extensions (Temperature Scaling, Stacking, Conformal, Adversarial, Stress, Ensemble Asymétrique)

**Correction principale :** `PreScaledDataset` remplace `VIXAmplitudeDataset` pour val/test — élimine le double-scaling qui causait F1_dir=0.325 (prédiction d'une seule classe).


## 0. Installation

In [ ]:
import sys
!{sys.executable} -m pip install -q arch pykalman hmmlearn shap xgboost xlsxwriter imbalanced-learn yfinance pandas_datareader statsmodels mapie
!{sys.executable} -m pip install -q torch-geometric 2>/dev/null || True
MAMBA_AVAILABLE = False

import os, time, random, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score,
                              precision_score, recall_score)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import mutual_info_classif
from xgboost import XGBClassifier

from imblearn.over_sampling import BorderlineSMOTE, SMOTE
from imblearn.combine import SMOTETomek

import matplotlib.pyplot as plt
import shap
from arch import arch_model
from pykalman import KalmanFilter
from hmmlearn import hmm as hmmlib

try:
    from mapie.classification import MapieClassifier
    MAPIE_AVAILABLE = True
except: MAPIE_AVAILABLE = False

try:
    from torch_geometric.nn import GATConv
    from torch_geometric.data import Data
    GNN_AVAILABLE = True
except:
    GNN_AVAILABLE = False

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device} | GNN: {GNN_AVAILABLE}")


## 1. Configuration

In [ ]:
CONFIG = {
    'seed': 42, 'start_date': '2012-01-01',
    'lookback': 21, 'horizons': [1, 3, 5, 7],
    'flat_thr': 0.003,
    'top_n_shap': 40, 'top_n_final': 80,
    'max_interaction_keep': 3000,
    'interaction_batch_size': 1500,
    'interaction_keep_per_batch': 500,
    'batch_size': 64, 'epochs': 60, 'lr': 3e-4,
    'weight_decay': 1e-4, 'dropout': 0.3,
    'class_quantiles': [0.25, 0.75],
    'class_labels': ['DOWN_FORT','DOWN_FAIBLE','UP_FAIBLE','UP_FORT'],
    'temperature_lr': 0.01, 'temperature_epochs': 100,
    'conformal_alpha': 0.10,
    'adv_val_threshold': 0.70,
}

TARGET_COL = 'VIX_Amplitude_Class'

YF_TICKERS = """
^GSPC ^IXIC ^VIX ^VXN ^OVX ^GVZ ^EVZ ^VVIX
^FTSE ^N225 ^HSI ^GDAXI ^STOXX50E
SPY QQQ TLT GLD USO HYG LQD
AAPL AMZN MSFT NVDA INTC QCOM XOM WMT MCD SBUX
MS COF BLK SCHW CLX CPB LMT NOC GD HON
CCI PSA EQIX NEE TXN PAYX LUV CMCSA
XLK XLF XLE XLV XLU XLB XLI XLY
""".split()

FRED_SERIES = {'NFCI':'NFCI','STLFSI':'STLFSI4','T10Y2Y':'T10Y2Y','EFFR':'EFFR'}

CRISIS_PERIODS = {
    'GFC_2008':      ('2008-09-01','2009-03-31'),
    'Euro_2011':     ('2011-07-01','2012-01-31'),
    'COVID_2020':    ('2020-02-20','2020-05-31'),
    'Fed_Hike_2022': ('2022-01-01','2022-12-31'),
    'SVB_2023':      ('2023-03-01','2023-05-31'),
}
REGIME_THRESHOLDS = {'calm': 18.0, 'stress': 25.0}
print("Config chargée.")


## 2. Chargement des données

In [ ]:
def load_data(start=CONFIG['start_date']):
    t0 = time.time()
    raw = yf.download(YF_TICKERS, start=start, auto_adjust=True, progress=False)['Close']
    raw.columns = [c.replace('^','IDX_').replace('-','_') for c in raw.columns]
    coverage = raw.notna().mean()
    raw = raw.loc[:, coverage >= 0.90].ffill().dropna(how='all')
    fred_frames = []
    for name, sid in FRED_SERIES.items():
        try:
            s = web.DataReader(sid,'fred',start).squeeze(); s.name=f'FRED_{name}'
            fred_frames.append(s)
        except Exception as e: print(f"  [WARN] FRED {sid}: {e}")
    if fred_frames:
        raw = pd.concat([raw, pd.concat(fred_frames,axis=1).reindex(raw.index,method='ffill')], axis=1)
    print(f"  Dataset: {raw.shape[0]}j × {raw.shape[1]} séries ({time.time()-t0:.1f}s)")
    return raw

df_raw = load_data()


## 3. Features Time-Series robustes

Version issue du notebook Enriched — gestion complète des NaN/inf à chaque étape.
EGARCH, Kalman (avec shift anti-leakage), HMM, Heston proxies + espérances conditionnelles,
VRP via HAR-RV, Jump Intensity, Hawkes Process.


In [ ]:
# ============================================================
# 3. Features Time-Series : EGARCH, Kalman, HMM, Heston, VRP, Jumps, Hawkes
# ============================================================

def build_ts_features(df_raw, train_end_idx):
    """
    Construit les features time-series.
    Tout fit est fait sur le train uniquement, puis appliqué au dataset complet.
    Version robuste contre NaN et inf.
    """
    t0 = time.time()
    feats = pd.DataFrame(index=df_raw.index)

    vix_col = 'IDX_VIX' if 'IDX_VIX' in df_raw.columns else [c for c in df_raw.columns if 'VIX' in c and 'VVIX' not in c][0]
    spx_col = [c for c in df_raw.columns if 'GSPC' in c or 'SPY' in c][0]

    vix = df_raw[vix_col].replace([np.inf, -np.inf], np.nan).astype(float).ffill().bfill()
    spx = df_raw[spx_col].replace([np.inf, -np.inf], np.nan).astype(float).ffill().bfill()

    vix_ret = np.log(vix / vix.shift(1)).replace([np.inf, -np.inf], np.nan).fillna(0)
    spx_ret = np.log(spx / spx.shift(1)).replace([np.inf, -np.inf], np.nan).fillna(0)

    # EGARCH
    try:
        spx_ret_train = spx_ret.iloc[:train_end_idx].replace([np.inf, -np.inf], np.nan).fillna(0) * 100
        am = arch_model(spx_ret_train, vol='EGARCH', p=1, q=1, dist='skewt', rescale=False)
        res_eg = am.fit(disp='off', show_warning=False)
        eg_full = res_eg.forecast(start=0, reindex=True)
        condvar = eg_full.variance.iloc[:, 0] / 10000
        condvar = condvar.reindex(df_raw.index, method='ffill').replace([np.inf, -np.inf], np.nan).ffill().bfill()
        feats['egarch_condvar'] = condvar
        feats['egarch_delta'] = condvar.diff()
        print(f"  EGARCH fit OK ({time.time() - t0:.1f}s)")
    except Exception as e:
        print(f"  [WARN] EGARCH: {e}")
        feats['egarch_condvar'] = np.nan
        feats['egarch_delta'] = np.nan

    # Kalman
    try:
        vix_clean = vix.replace([np.inf, -np.inf], np.nan).interpolate('linear').ffill().bfill().astype(float)
        if not np.isfinite(vix_clean.values).all():
            finite_mean = np.nanmean(vix_clean.replace([np.inf, -np.inf], np.nan).values)
            vix_clean = vix_clean.replace([np.inf, -np.inf], np.nan).fillna(finite_mean)

        train_vix_clean = vix_clean.iloc[:train_end_idx].values.reshape(-1, 1)
        full_vix_clean = vix_clean.values.reshape(-1, 1)

        kf = KalmanFilter(
            transition_matrices=np.array([[1.0]]),
            observation_matrices=np.array([[1.0]]),
            initial_state_mean=np.array([float(vix_clean.iloc[0])]),
            initial_state_covariance=np.array([[1.0]]),
            transition_covariance=np.array([[1e-3]]),
            observation_covariance=np.array([[1e-2]]),
            em_vars=['transition_covariance', 'observation_covariance']
        )
        kf = kf.em(train_vix_clean, n_iter=20)
        sm, _ = kf.filter(full_vix_clean)
        ss, _ = kf.smooth(full_vix_clean)

        kalman_filtered = pd.Series(sm[:, 0], index=df_raw.index)
        kalman_smooth = pd.Series(ss[:, 0], index=df_raw.index)
        feats['kalman_residual'] = (vix_clean - kalman_filtered).shift(1).replace([np.inf, -np.inf], np.nan)
        feats['kalman_innovation'] = (vix_clean - kalman_smooth.shift(1)).shift(1).replace([np.inf, -np.inf], np.nan)
        print(f"  Kalman fit OK ({time.time() - t0:.1f}s)")
    except Exception as e:
        print(f"  [WARN] Kalman: {e}")
        feats['kalman_residual'] = np.nan
        feats['kalman_innovation'] = np.nan

    # HMM
    try:
        rv5 = vix_ret.pow(2).rolling(5, min_periods=3).mean()
        mu = vix.iloc[:train_end_idx].mean()
        sd = vix.iloc[:train_end_idx].std()
        if not np.isfinite(sd) or sd <= 1e-12:
            sd = 1.0
        vix_n = (vix - mu) / sd
        X_hmm = pd.DataFrame({'ret': vix_ret, 'vol5': np.sqrt(rv5), 'level': vix_n}, index=df_raw.index)
        X_hmm = X_hmm.replace([np.inf, -np.inf], np.nan).dropna()
        train_dates = df_raw.index[:train_end_idx]
        X_tr_df = X_hmm.loc[X_hmm.index.isin(train_dates)]
        if len(X_tr_df) < 50:
            raise ValueError('Pas assez de données propres pour HMM.')

        model_hmm = hmmlib.GaussianHMM(n_components=2, covariance_type='full', n_iter=200, random_state=SEED)
        model_hmm.fit(X_tr_df.values)
        states_tr = model_hmm.predict(X_tr_df.values)
        rv5_train = rv5.reindex(X_tr_df.index)
        state_vols = []
        for s in range(2):
            vals = rv5_train.values[states_tr == s]
            vals = vals[np.isfinite(vals)]
            state_vols.append(np.nanmean(vals) if len(vals) else -np.inf)
        stress_st = int(np.argmax(state_vols))
        proba_full = model_hmm.predict_proba(X_hmm.values)
        states_full = model_hmm.predict(X_hmm.values)
        feats['hmm_p_stress'] = pd.Series(proba_full[:, stress_st], index=X_hmm.index).reindex(df_raw.index)
        feats['hmm_state'] = pd.Series(states_full, index=X_hmm.index).reindex(df_raw.index)
        print(f"  HMM fit OK ({time.time() - t0:.1f}s)")
    except Exception as e:
        print(f"  [WARN] HMM: {e}")
        feats['hmm_p_stress'] = np.nan
        feats['hmm_state'] = np.nan

    # Heston proxies
    v0 = (vix / 100).pow(2)
    theta = vix_ret.pow(2).rolling(60, min_periods=30).mean()
    vvix_col = [c for c in df_raw.columns if 'VVIX' in c]
    if vvix_col:
        xi = df_raw[vvix_col[0]].replace([np.inf, -np.inf], np.nan).astype(float).ffill().bfill() / 100
    else:
        xi = vix_ret.rolling(20, min_periods=10).std()
    rho = vix_ret.rolling(30, min_periods=15).corr(spx_ret)

    def rolling_kappa(series, w=252):
        kappa = pd.Series(np.nan, index=series.index)
        s = series.replace([np.inf, -np.inf], np.nan).ffill().bfill()
        for i in range(w, len(s)):
            y = s.iloc[i-w+1:i+1].values
            x = s.iloc[i-w:i].values
            try:
                if not np.isfinite(x).all() or not np.isfinite(y).all():
                    continue
                beta = np.corrcoef(x, y)[0, 1]
                if np.isfinite(beta) and 0 < abs(beta) < 0.9999:
                    hl = -np.log(2) / np.log(abs(beta))
                    if np.isfinite(hl) and hl > 0:
                        kappa.iloc[i] = np.log(2) / hl
            except Exception:
                pass
        return kappa.replace([np.inf, -np.inf], np.nan)

    kappa = rolling_kappa(vix)
    feats['heston_v0'] = v0
    feats['heston_theta'] = theta
    feats['heston_xi'] = xi
    feats['heston_rho'] = rho
    feats['heston_kappa'] = kappa
    feats['heston_feller'] = (2 * kappa * theta) / xi.pow(2).replace(0, np.nan)
    feats['heston_v0_minus_theta'] = v0 - theta

    for h in CONFIG['horizons']:
        ev = theta + (v0 - theta) * np.exp(-kappa * h)
        ev = ev.replace([np.inf, -np.inf], np.nan)
        feats[f'heston_ev_h{h}'] = ev
        feats[f'heston_spread_h{h}'] = v0 - ev
        feats[f'heston_vol_h{h}'] = np.sqrt(ev.clip(lower=0)) * 100

    # VRP HAR-RV
    try:
        import statsmodels.api as sm
        rv1d = vix_ret.pow(2)
        rv5d = rv1d.rolling(5, min_periods=3).mean()
        rv22d = rv1d.rolling(22, min_periods=10).mean()
        rv_target = rv1d.shift(-22).rolling(22, min_periods=11).mean()
        har_df = pd.DataFrame({'rv1': rv1d, 'rv5': rv5d, 'rv22': rv22d, 'y': rv_target}, index=df_raw.index)
        har_df = har_df.replace([np.inf, -np.inf], np.nan).dropna()
        train_dates = df_raw.index[:train_end_idx]
        har_train = har_df.loc[har_df.index.isin(train_dates)]
        if len(har_train) < 50:
            raise ValueError('Pas assez de données propres pour HAR-RV.')
        Xh = sm.add_constant(har_train[['rv1', 'rv5', 'rv22']], has_constant='add')
        har_model = sm.OLS(har_train['y'], Xh).fit()
        Xfull = sm.add_constant(har_df[['rv1', 'rv5', 'rv22']], has_constant='add')
        rv_pred = har_model.predict(Xfull).reindex(df_raw.index)
        feats['VRP'] = (vix / 100).pow(2) - rv_pred
        vrp_train = feats['VRP'].iloc[:train_end_idx]
        vrp_sd = vrp_train.std()
        if not np.isfinite(vrp_sd) or vrp_sd <= 1e-12:
            vrp_sd = 1.0
        feats['VRP_zscore'] = (feats['VRP'] - vrp_train.mean()) / vrp_sd
    except Exception as e:
        print(f"  [WARN] VRP: {e}")
        feats['VRP'] = np.nan
        feats['VRP_zscore'] = np.nan

    # Jumps et Hawkes simplifié
    sigma_60 = vix_ret.rolling(60, min_periods=30).std()
    is_jump = (vix_ret.abs() > 3 * sigma_60).astype(float)
    feats['jump_intensity_20d'] = is_jump.rolling(20, min_periods=10).mean()
    feats['jump_intensity_60d'] = is_jump.rolling(60, min_periods=30).mean()

    sigma_hw = vix_ret.rolling(30, min_periods=15).std()
    jump_times = vix_ret.index[vix_ret.abs() > 2 * sigma_hw]
    hawkes = pd.Series(0.0, index=vix_ret.index)
    for i, t in enumerate(vix_ret.index):
        past = jump_times[jump_times < t]
        if len(past) > 0:
            days_since = np.array([(t - tj).days for tj in past])
            hawkes.iloc[i] = 0.3 + 0.3 * np.sum(np.exp(-0.1 * days_since))
        else:
            hawkes.iloc[i] = 0.3
    feats['hawkes_intensity'] = hawkes
    sd_hw = hawkes.iloc[:train_end_idx].std()
    if not np.isfinite(sd_hw) or sd_hw <= 1e-12:
        sd_hw = 1.0
    feats['hawkes_zscore'] = (hawkes - hawkes.iloc[:train_end_idx].mean()) / sd_hw

    feats = feats.replace([np.inf, -np.inf], np.nan)
    print(f"  TS features OK ({time.time() - t0:.1f}s)")
    return feats


## 4. Feature Engineering avancé

In [ ]:
# ============================================================
# 5. Feature engineering avancé
# ============================================================

def build_advanced_features(df_raw, vix_series, spx_series, train_end_idx):
    feats = pd.DataFrame(index=df_raw.index)
    vix = vix_series.replace([np.inf, -np.inf], np.nan).ffill().bfill()
    spx = spx_series.replace([np.inf, -np.inf], np.nan).ffill().bfill()
    vix_ret = np.log(vix / vix.shift(1)).replace([np.inf, -np.inf], np.nan)
    spx_ret = np.log(spx / spx.shift(1)).replace([np.inf, -np.inf], np.nan)

    feats['vix_vol_of_vol_5d'] = vix_ret.rolling(5, min_periods=3).std()
    feats['vix_vol_of_vol_10d'] = vix_ret.rolling(10, min_periods=5).std()
    feats['vix_momentum_2d'] = vix.pct_change(2)
    feats['vix_momentum_3d'] = vix.pct_change(3)
    feats['vix_acceleration'] = vix_ret - vix_ret.shift(3)
    feats['vix_erratic_ratio'] = vix_ret.abs().rolling(5, min_periods=3).max() / vix_ret.abs().rolling(5, min_periods=3).mean().replace(0, np.nan)
    feats['vix_vol_ratio_5_60'] = vix_ret.rolling(5, min_periods=3).std() / vix_ret.rolling(60, min_periods=30).std().replace(0, np.nan)
    feats['spx_vol_5d'] = spx_ret.rolling(5, min_periods=3).std()
    feats['spx_abs_ret_max_5d'] = spx_ret.abs().rolling(5, min_periods=3).max()
    feats['spx_momentum_3d'] = spx.pct_change(3)
    feats['vix_spx_corr_30d'] = vix_ret.rolling(30, min_periods=15).corr(spx_ret)

    for w in [5, 10, 20]:
        ma = vix.rolling(w, min_periods=max(2, w // 2)).mean()
        feats[f'vix_vs_ma{w}'] = (vix - ma) / ma.replace(0, np.nan)
        feats[f'vix_zscore_{w}d'] = (vix - ma) / vix.rolling(w, min_periods=max(2, w // 2)).std().replace(0, np.nan)

    for h in [1, 2, 3, 5, 10, 20]:
        feats[f'vix_ret_{h}d'] = vix.pct_change(h)
        feats[f'spx_ret_{h}d'] = spx.pct_change(h)

    def rsi(series, n=14):
        delta = series.diff()
        gain = delta.clip(lower=0).rolling(n).mean()
        loss = (-delta.clip(upper=0)).rolling(n).mean()
        rs = gain / loss.replace(0, np.nan)
        return 100 - 100 / (1 + rs)

    feats['vix_rsi_14'] = rsi(vix, 14)
    feats['spx_rsi_14'] = rsi(spx, 14)

    for w in [10, 20]:
        ma = vix.rolling(w).mean()
        std = vix.rolling(w).std()
        feats[f'boll_width_{w}d'] = (2 * std) / ma.replace(0, np.nan)

    ema12 = vix.ewm(span=12).mean()
    ema26 = vix.ewm(span=26).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9).mean()
    feats['vix_macd'] = macd
    feats['vix_macd_signal'] = signal
    feats['vix_macd_hist'] = macd - signal
    feats['vix_roll_skew_20d'] = vix_ret.rolling(20).skew()
    feats['vix_roll_kurt_20d'] = vix_ret.rolling(20).kurt()

    for w in [20, 60]:
        roll_max = vix.rolling(w).max()
        roll_min = vix.rolling(w).min()
        feats[f'vix_dist_max_{w}d'] = (vix - roll_max) / roll_max.replace(0, np.nan)
        feats[f'vix_dist_min_{w}d'] = (vix - roll_min) / roll_min.replace(0, np.nan)

    roll_high = spx.rolling(252, min_periods=126).max()
    feats['spx_drawdown_252d'] = (spx - roll_high) / roll_high.replace(0, np.nan)

    # Z-score sur train uniquement.
    original_cols = list(feats.columns)
    for col in original_cols:
        mu = feats[col].iloc[:train_end_idx].mean()
        sd = feats[col].iloc[:train_end_idx].std()
        if np.isfinite(sd) and sd > 1e-8:
            feats[f'{col}_z'] = (feats[col] - mu) / sd

    return feats.replace([np.inf, -np.inf], np.nan)


## 5. Cible amplitude 4 classes (quantiles conditionnels par régime)

In [ ]:
# ============================================================
# 4. Cible d'amplitude
# ============================================================

def build_amplitude_target(vix_series, horizon, train_end_idx, regime_series=None):
    vix = vix_series.replace([np.inf, -np.inf], np.nan).ffill().bfill().copy()
    ret = (vix.shift(-horizon) / vix) - 1
    ret = ret.replace([np.inf, -np.inf], np.nan).dropna()

    flat_mask = ret.abs() < CONFIG['flat_thr']
    ret = ret.loc[~flat_mask]

    vix_train = vix.iloc[:train_end_idx]
    calm_thr = vix_train.quantile(0.33)
    stress_thr = vix_train.quantile(0.67)

    regime = pd.Series('NORMAL', index=ret.index)
    regime[vix.reindex(ret.index) < calm_thr] = 'CALM'
    regime[vix.reindex(ret.index) >= stress_thr] = 'STRESS'

    ret_train = ret.iloc[:train_end_idx]
    thresholds = {}
    for reg in ['CALM', 'NORMAL', 'STRESS']:
        sub = ret_train[regime.iloc[:train_end_idx] == reg]
        if len(sub) >= 20:
            thresholds[reg] = (sub.quantile(0.25), sub.quantile(0.75))
        else:
            thresholds[reg] = (ret_train.quantile(0.25), ret_train.quantile(0.75))

    def classify(r, reg):
        q25, q75 = thresholds.get(reg, (0, 0))
        if r < q25:
            return 0
        if r < 0:
            return 1
        if r < q75:
            return 2
        return 3

    target = pd.Series([classify(r, regime[i]) for i, r in ret.items()], index=ret.index, name=TARGET_COL)
    print(f"  [Target h={horizon}j] {len(target)} obs, dist: {target.value_counts().to_dict()}")
    return target, regime, ret, thresholds


## 6. Interactions et sélection SHAP

Interactions sur toutes les features candidates (traitement par batch pour économiser la mémoire).
Mutual Information comme scoring intermédiaire par batch, SHAP XGBoost pour la sélection finale.


In [ ]:
# ============================================================
# 6. Interactions sur toutes les features candidates + sélection finale
# ============================================================

def _make_interaction_series(df, fi, fj, op, rolling_w=20, eps=1e-8):
    si = df[fi]
    sj = df[fj]
    if op == 'div':
        return si / sj.where(sj.abs() >= eps, np.nan)
    if op == 'minus':
        return si - sj
    if op == 'prod':
        return si * sj
    if op == 'zrel':
        diff = si - sj
        rs = diff.rolling(rolling_w, min_periods=max(2, rolling_w // 2)).std()
        return diff / rs.replace(0, np.nan)
    if op == 'macross':
        mai = si.rolling(rolling_w, min_periods=max(2, rolling_w // 2)).mean()
        maj = sj.rolling(rolling_w, min_periods=max(2, rolling_w // 2)).mean()
        return mai / maj.where(maj.abs() >= eps, np.nan)
    if op == 'ret5x':
        return si.pct_change(5) * sj
    raise ValueError(f'Opération inconnue : {op}')


def apply_interaction_specs(df, specs, rolling_w=20, eps=1e-8):
    cols = {}
    for fi, fj, op, name in specs:
        if fi not in df.columns or fj not in df.columns:
            continue
        try:
            cols[name] = _make_interaction_series(df, fi, fj, op, rolling_w=rolling_w, eps=eps)
        except Exception:
            pass
    if not cols:
        return pd.DataFrame(index=df.index)
    out = pd.DataFrame(cols, index=df.index).replace([np.inf, -np.inf], np.nan).dropna(axis=1, how='all')
    return out


def generate_interactions_all_features_batched(
    df,
    y,
    feature_cols,
    max_keep=3000,
    batch_size=1500,
    keep_per_batch=500,
    rolling_w=20,
    eps=1e-8,
    random_state=SEED
):
    feature_cols = [c for c in feature_cols if c in df.columns]
    ops = ['div', 'minus', 'prod', 'zrel', 'macross', 'ret5x']
    selected_chunks = []
    selected_specs = []
    selected_scores = []
    batch_cols = {}
    batch_specs = []
    y_arr = np.asarray(y).astype(int)

    total_pairs = len(feature_cols) * (len(feature_cols) - 1) // 2
    total_raw = total_pairs * len(ops)
    print(f"  [INTER ALL] {len(feature_cols)} features candidates")
    print(f"  [INTER ALL] Interactions théoriques : {total_raw}")

    def flush_batch(batch_cols, batch_specs):
        if not batch_cols:
            return None, [], pd.Series(dtype=float)
        tmp = pd.DataFrame(batch_cols, index=df.index)
        tmp = tmp.replace([np.inf, -np.inf], np.nan).dropna(axis=1, how='all')
        if tmp.shape[1] == 0:
            return None, [], pd.Series(dtype=float)
        tmp_clean = tmp.fillna(0)
        vals = np.nan_to_num(tmp_clean.values, nan=0.0, posinf=0.0, neginf=0.0)
        try:
            mi = mutual_info_classif(vals, y_arr, random_state=random_state)
            scores = pd.Series(mi, index=tmp_clean.columns).sort_values(ascending=False)
        except Exception:
            scores = tmp_clean.var().sort_values(ascending=False)
        keep_cols = scores.head(min(keep_per_batch, len(scores))).index.tolist()
        kept_specs = [spec for spec in batch_specs if spec[3] in keep_cols]
        return tmp_clean[keep_cols], kept_specs, scores.loc[keep_cols]

    counter = 0
    for i in range(len(feature_cols)):
        fi = feature_cols[i]
        for j in range(i + 1, len(feature_cols)):
            fj = feature_cols[j]
            for op in ops:
                name = f"{fi}__{op}__{fj}"
                try:
                    batch_cols[name] = _make_interaction_series(df, fi, fj, op, rolling_w=rolling_w, eps=eps)
                    batch_specs.append((fi, fj, op, name))
                    counter += 1
                except Exception:
                    continue
                if len(batch_cols) >= batch_size:
                    kept_df, kept_specs, kept_scores = flush_batch(batch_cols, batch_specs)
                    if kept_df is not None:
                        selected_chunks.append(kept_df)
                        selected_specs.extend(kept_specs)
                        selected_scores.append(kept_scores)
                    batch_cols = {}
                    batch_specs = []
                    print(f"    [INTER ALL] {counter}/{total_raw} interactions traitées")

    kept_df, kept_specs, kept_scores = flush_batch(batch_cols, batch_specs)
    if kept_df is not None:
        selected_chunks.append(kept_df)
        selected_specs.extend(kept_specs)
        selected_scores.append(kept_scores)

    if not selected_chunks:
        return pd.DataFrame(index=df.index), [], pd.Series(dtype=float)

    interactions = pd.concat(selected_chunks, axis=1)
    interactions = interactions.loc[:, ~interactions.columns.duplicated()]
    interactions = interactions.replace([np.inf, -np.inf], np.nan).fillna(0)

    all_scores = pd.concat(selected_scores) if selected_scores else pd.Series(dtype=float)
    all_scores = all_scores[~all_scores.index.duplicated(keep='first')].sort_values(ascending=False)

    if max_keep is not None and interactions.shape[1] > max_keep:
        final_cols = [c for c in all_scores.head(max_keep).index.tolist() if c in interactions.columns]
        interactions = interactions[final_cols]
        selected_specs = [spec for spec in selected_specs if spec[3] in final_cols]
        all_scores = all_scores.loc[final_cols]

    print(f"  [INTER ALL] Interactions retenues : {interactions.shape[1]}")
    return interactions, selected_specs, all_scores


def shap_select_features(X_train, y_train, top_n, label=''):
    try:
        from xgboost import XGBClassifier
        from sklearn.preprocessing import LabelEncoder
        X_clean = X_train.copy().replace([np.inf, -np.inf], np.nan).fillna(0)
        bad_cols = [c for c in X_clean.columns if not np.isfinite(X_clean[c].values).all()]
        if bad_cols:
            X_clean = X_clean.drop(columns=bad_cols)
        le = LabelEncoder()
        y_enc = le.fit_transform(y_train)
        pilot = XGBClassifier(
            n_estimators=80,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            eval_metric='mlogloss',
            objective='multi:softprob',
            random_state=SEED,
            n_jobs=-1
        )
        pilot.fit(X_clean.values, y_enc)
        expl = shap.TreeExplainer(pilot)
        sv = expl.shap_values(X_clean.values[:500])
        if isinstance(sv, list):
            arr = np.mean([np.abs(s) for s in sv], axis=0)
        elif np.array(sv).ndim == 3:
            arr = np.abs(sv).mean(axis=2)
        else:
            arr = np.abs(sv)
        scores = pd.Series(arr.mean(axis=0), index=X_clean.columns)
        top = scores.sort_values(ascending=False).head(top_n).index.tolist()
        if label:
            print(f"  [SHAP {label}] top-{len(top)}/{len(scores)}")
        return top, scores.sort_values(ascending=False)
    except Exception as e:
        print(f"  [WARN] SHAP indisponible, fallback mutual information: {e}")
        X_clean = X_train.copy().replace([np.inf, -np.inf], np.nan).fillna(0)
        vals = np.nan_to_num(X_clean.values, nan=0.0, posinf=0.0, neginf=0.0)
        mi = mutual_info_classif(vals, y_train, random_state=SEED)
        scores = pd.Series(mi, index=X_clean.columns).sort_values(ascending=False)
        return scores.head(top_n).index.tolist(), scores


## 7. Datasets PyTorch — Correction du double-scaling

**`PreScaledDataset`** : utilisé pour val/test — reçoit des données déjà scalées, 
ne refitte pas le scaler. Corrige le bug principal (F1_dir=0.325).

**`VIXAmplitudeDataset`** : utilisé uniquement pour le train rééchantillonné 
(fit du scaler sur le train SMOTE).


In [ ]:
# VIXAmplitudeDataset : FIT le scaler (train SMOTE uniquement)
class VIXAmplitudeDataset(Dataset):
    def __init__(self, data, feature_cols, target_col=TARGET_COL,
                 lookback=CONFIG['lookback'], scaler=None):
        self.lookback = lookback
        X = data[feature_cols].copy().replace([np.inf,-np.inf],np.nan).fillna(0).astype(float)
        y = data[target_col].values.astype(int)
        self.scaler = scaler if scaler is not None else RobustScaler()
        self.X = (self.scaler.fit_transform(X) if scaler is None
                  else self.scaler.transform(X))
        self.X = np.nan_to_num(self.X, nan=0., posinf=0., neginf=0.)
        self.y = y
        self.n_features = self.X.shape[1]
    def __len__(self): return max(0, len(self.X) - self.lookback)
    def __getitem__(self, i):
        return (torch.tensor(self.X[i:i+self.lookback], dtype=torch.float32),
                torch.tensor(self.y[i+self.lookback], dtype=torch.long))


# PreScaledDataset : TRANSFORM seulement (val/test — jamais refitter)
class PreScaledDataset(Dataset):
    """
    Reçoit X déjà scalé (numpy array) — zéro re-scaling.
    Corrige le double-scaling qui causait F1_dir=0.325 (prédiction d'une seule classe).

    La séquence [i:i+lookback] → label [i+lookback] garantit la causalité :
    le modèle voit les lookback jours AVANT la date à prédire.
    """
    def __init__(self, X, y, lookback=CONFIG['lookback']):
        self.X = np.nan_to_num(X.astype(np.float32), nan=0., posinf=0., neginf=0.)
        self.y = y.astype(np.int64)
        self.lookback = lookback
    def __len__(self): return max(0, len(self.X) - self.lookback)
    def __getitem__(self, i):
        return (torch.tensor(self.X[i:i+self.lookback]),
                torch.tensor(self.y[i+self.lookback]))


## 8. Architectures DL : LSTM, TCN, Transformer, CNN-LSTM, N-BEATS, TFT, Mamba

In [ ]:
# ============================================================
# 8. Architectures Deep Learning
# ============================================================

class VIX_LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.GELU(),
            nn.Linear(32, n_classes)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.norm(out[:, -1, :]))

class TCNBlock(nn.Module):
    def __init__(self, n_in, n_out, kernel_size, dilation, dropout=0.2):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.conv = nn.utils.weight_norm(nn.Conv1d(n_in, n_out, kernel_size, padding=pad, dilation=dilation))
        self.drop = nn.Dropout(dropout)
        self.skip = nn.Conv1d(n_in, n_out, 1) if n_in != n_out else None
        self.act = nn.GELU()
    def forward(self, x):
        conv_out = self.conv(x)
        if self.conv.padding[0] > 0:
            conv_out = conv_out[:, :, :-self.conv.padding[0]]
        out = self.act(self.drop(conv_out))
        res = x if self.skip is None else self.skip(x)
        return self.act(out + res)

class VIX_TCN(nn.Module):
    def __init__(self, input_dim, channels=[32, 64, 128], kernel_size=3, dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        layers = []
        in_ch = input_dim
        for i, ch in enumerate(channels):
            layers.append(TCNBlock(in_ch, ch, kernel_size, dilation=2 ** i, dropout=dropout))
            in_ch = ch
        self.net = nn.Sequential(*layers)
        self.fc = nn.Sequential(nn.Linear(channels[-1], 64), nn.GELU(), nn.Dropout(dropout), nn.Linear(64, n_classes))
    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.net(x)
        return self.fc(x.mean(dim=2))

class VIX_Transformer(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=4, num_layers=3, dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        self.proj = nn.Linear(input_dim, d_model)
        self.pos_emb = nn.Embedding(CONFIG['lookback'], d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4, dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model, 64), nn.GELU(), nn.Dropout(dropout), nn.Linear(64, n_classes))
    def forward(self, x):
        B, T, _ = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x = self.proj(x) + self.pos_emb(pos)
        out = self.encoder(x)
        return self.fc(out[:, -1, :])

class VIX_CNNLSTM(nn.Module):
    def __init__(self, input_dim, conv_filters=64, lstm_hidden=128, kernel_size=3, dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, conv_filters, kernel_size=kernel_size, padding='same'),
            nn.GELU(), nn.BatchNorm1d(conv_filters), nn.Dropout(dropout)
        )
        self.lstm = nn.LSTM(conv_filters, lstm_hidden, batch_first=True, num_layers=2, dropout=dropout)
        self.fc = nn.Sequential(nn.Linear(lstm_hidden, 64), nn.GELU(), nn.Dropout(dropout), nn.Linear(64, n_classes))
    def forward(self, x):
        x = self.conv(x.transpose(1, 2)).transpose(1, 2)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class NBeatsBlock(nn.Module):
    def __init__(self, input_size, theta_size, hidden_size=256):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, theta_size)
        )
    def forward(self, x):
        return self.fc(x)

class VIX_NBeats(nn.Module):
    def __init__(self, input_dim, lookback, n_blocks=3, hidden_size=256, dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        self.blocks = nn.ModuleList([NBeatsBlock(input_dim * lookback, hidden_size, hidden_size) for _ in range(n_blocks)])
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, n_classes)
    def forward(self, x):
        residual = x.reshape(x.size(0), -1)
        out = None
        for block in self.blocks:
            theta = self.drop(block(residual))
            out = theta if out is None else out + theta
        return self.fc(out)

class GatedLinearUnit(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, out_dim)
        self.gate = nn.Linear(in_dim, out_dim)
    def forward(self, x):
        return self.fc(x) * torch.sigmoid(self.gate(x))

class VIX_TFT(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=4, lstm_layers=2, dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        self.vsn = nn.Sequential(nn.Linear(input_dim, d_model), nn.GELU(), nn.Linear(d_model, input_dim), nn.Softmax(dim=-1))
        self.input_proj = nn.Linear(input_dim, d_model)
        self.lstm = nn.LSTM(d_model, d_model, num_layers=lstm_layers, batch_first=True, dropout=dropout)
        self.lstm_norm = nn.LayerNorm(d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=d_model * 2, dropout=dropout, batch_first=True, norm_first=True)
        self.attention = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.glu = GatedLinearUnit(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Sequential(nn.Linear(d_model, 64), nn.GELU(), nn.Dropout(dropout), nn.Linear(64, n_classes))
    def forward(self, x):
        weights = self.vsn(x.mean(dim=1, keepdim=True))
        x = x * weights
        x = self.input_proj(x)
        lstm_out, _ = self.lstm(x)
        lstm_out = self.lstm_norm(lstm_out + x)
        attn_out = self.attention(lstm_out)
        out = self.norm(self.glu(attn_out) + attn_out)
        return self.fc(out[:, -1, :])

print('Architectures définies : LSTM, TCN, Transformer, CNN-LSTM, N-BEATS, TFT')



# ── Mamba (SSM simplifié) ─────────────────────────────────────────────────────
class SSMLayer(nn.Module):
    def __init__(self, d, ks=21):
        super().__init__()
        self.kernel = nn.Parameter(torch.randn(d,1,ks)*0.01)
        self.norm = nn.LayerNorm(d)
        self.ig = nn.Linear(d,d); self.og = nn.Linear(d,d)
    def forward(self, x):
        B,L,D = x.shape
        xp = F.pad(x.transpose(1,2),(self.kernel.shape[2]-1,0))
        k  = torch.softmax(self.kernel, dim=-1)
        out = F.conv1d(xp, k, groups=D).transpose(1,2)
        out = out * torch.sigmoid(self.ig(x))
        out = self.norm(out + x)
        return out * torch.sigmoid(self.og(out))

class VIX_Mamba(nn.Module):
    def __init__(self, input_dim, d_model=128, n_layers=4,
                 dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        self.proj   = nn.Linear(input_dim, d_model)
        self.layers = nn.ModuleList([SSMLayer(d_model) for _ in range(n_layers)])
        self.drop   = nn.Dropout(dropout); self.norm = nn.LayerNorm(d_model)
        self.fc     = nn.Sequential(nn.Linear(d_model,64),nn.GELU(),
                                     nn.Dropout(dropout),nn.Linear(64,n_classes))
    def forward(self, x):
        x = self.proj(x)
        for l in self.layers: x = self.drop(l(x))
        return self.fc(self.norm(x[:,-1,:]))

print("7 architectures : LSTM, TCN, Transformer, CNN-LSTM, N-BEATS, TFT, Mamba")


## 9. Focal Loss, entraînement, métriques complètes

In [ ]:
# ============================================================
# 9. Entraînement et métriques complètes
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.1, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.ls = label_smoothing
        self.reduction = reduction
    def forward(self, logits, targets):
        n_cls = logits.size(1)
        one_hot = torch.zeros_like(logits).scatter_(1, targets.unsqueeze(1), 1)
        smooth = one_hot * (1 - self.ls) + self.ls / n_cls
        log_prob = torch.log_softmax(logits, dim=1)
        prob = log_prob.exp()
        alpha_t = self.alpha.to(logits.device)[targets] if self.alpha is not None else 1.0
        focal_weight = (1 - prob) ** self.gamma
        loss = -(alpha_t.unsqueeze(1) * focal_weight * smooth * log_prob).sum(dim=1)
        return loss.mean() if self.reduction == 'mean' else loss.sum()


def compute_class_weights(y_train, n_classes=4):
    counts = np.bincount(y_train, minlength=n_classes)
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum() * n_classes
    return torch.tensor(weights, dtype=torch.float32)


def train_model(model, train_loader, val_loader, class_weights=None, epochs=CONFIG['epochs'], lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'], label='Modèle', patience=10):
    criterion = FocalLoss(gamma=2.0, alpha=class_weights, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    best_val_loss = float('inf')
    best_state = None
    wait = 0
    history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
    t0 = time.time()

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        scheduler.step()

        model.eval()
        val_loss = 0.0
        val_preds, val_targets = [], []
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device), by.to(device)
                logits = model(bx)
                val_loss += criterion(logits, by).item()
                val_preds.extend(logits.argmax(1).cpu().numpy())
                val_targets.extend(by.cpu().numpy())
        val_loss_avg = val_loss / max(1, len(val_loader))
        train_loss_avg = train_loss / max(1, len(train_loader))
        val_f1 = f1_score(val_targets, val_preds, average='macro', zero_division=0) if len(val_targets) else 0
        history['train_loss'].append(train_loss_avg)
        history['val_loss'].append(val_loss_avg)
        history['val_f1'].append(val_f1)

        if val_loss_avg < best_val_loss:
            best_val_loss = val_loss_avg
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  [{label}] Early stop @ epoch {epoch + 1} ({time.time() - t0:.1f}s)")
                break
        if (epoch + 1) % 10 == 0:
            print(f"  [{label}] ep {epoch + 1}/{epochs} | val_f1={val_f1:.4f} | {time.time() - t0:.1f}s")

    if best_state:
        model.load_state_dict(best_state)
    return history


def safe_auc_multiclass(y_true, y_prob, n_classes=4):
    try:
        if y_prob.ndim != 2 or y_prob.shape[1] != n_classes or len(np.unique(y_true)) < 2:
            return np.nan
        return roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro', labels=list(range(n_classes)))
    except Exception:
        return np.nan


def safe_auc_binary(y_true_binary, y_score):
    try:
        if len(np.unique(y_true_binary)) < 2:
            return np.nan
        return roc_auc_score(y_true_binary, y_score)
    except Exception:
        return np.nan


def compute_metrics_from_arrays(y_true, y_pred, y_prob, label=''):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_prob = np.asarray(y_prob)

    metrics = {
        'Acc_4cls': accuracy_score(y_true, y_pred),
        'Precision_4cls_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'Precision_4cls_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall_4cls_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'Recall_4cls_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1_4cls_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'F1_4cls_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'AUC_4cls_ovr_macro': safe_auc_multiclass(y_true, y_prob, n_classes=4),
    }
    metrics['F1_4cls'] = metrics['F1_4cls_macro']

    y_true_dir = np.where(np.isin(y_true, [2, 3]), 1, 0)
    y_pred_dir = np.where(np.isin(y_pred, [2, 3]), 1, 0)
    y_prob_up = y_prob[:, 2] + y_prob[:, 3] if y_prob.ndim == 2 and y_prob.shape[1] >= 4 else np.full(len(y_true), np.nan)

    metrics.update({
        'Acc_dir': accuracy_score(y_true_dir, y_pred_dir),
        'Precision_dir_macro': precision_score(y_true_dir, y_pred_dir, average='macro', zero_division=0),
        'Precision_UP': precision_score(y_true_dir, y_pred_dir, pos_label=1, average='binary', zero_division=0),
        'Precision_DOWN': precision_score(y_true_dir, y_pred_dir, pos_label=0, average='binary', zero_division=0),
        'Recall_dir_macro': recall_score(y_true_dir, y_pred_dir, average='macro', zero_division=0),
        'Recall_UP': recall_score(y_true_dir, y_pred_dir, pos_label=1, average='binary', zero_division=0),
        'Recall_DOWN': recall_score(y_true_dir, y_pred_dir, pos_label=0, average='binary', zero_division=0),
        'F1_dir': f1_score(y_true_dir, y_pred_dir, average='macro', zero_division=0),
        'F1_UP': f1_score(y_true_dir, y_pred_dir, pos_label=1, average='binary', zero_division=0),
        'F1_DOWN': f1_score(y_true_dir, y_pred_dir, pos_label=0, average='binary', zero_division=0),
        'AUC_dir': safe_auc_binary(y_true_dir, y_prob_up),
    })

    up_idx = np.where(np.isin(y_true, [2, 3]))[0]
    dn_idx = np.where(np.isin(y_true, [0, 1]))[0]
    if len(up_idx) > 0:
        yt_up = np.where(y_true[up_idx] == 3, 1, 0)
        yp_up = np.where(y_pred[up_idx] == 3, 1, 0)
        metrics['Acc_UP_sub'] = accuracy_score(yt_up, yp_up)
        metrics['Precision_UP_FORT'] = precision_score(yt_up, yp_up, zero_division=0)
        metrics['Recall_UP_FORT'] = recall_score(yt_up, yp_up, zero_division=0)
        metrics['F1_UP_FORT'] = f1_score(yt_up, yp_up, zero_division=0)
        if y_prob.ndim == 2 and y_prob.shape[1] >= 4:
            prob_up_fort_cond = y_prob[up_idx, 3] / np.clip(y_prob[up_idx, 2] + y_prob[up_idx, 3], 1e-8, None)
            metrics['AUC_UP_FORT'] = safe_auc_binary(yt_up, prob_up_fort_cond)
    if len(dn_idx) > 0:
        yt_dn = np.where(y_true[dn_idx] == 0, 1, 0)
        yp_dn = np.where(y_pred[dn_idx] == 0, 1, 0)
        metrics['Acc_DOWN_sub'] = accuracy_score(yt_dn, yp_dn)
        metrics['Precision_DOWN_FORT'] = precision_score(yt_dn, yp_dn, zero_division=0)
        metrics['Recall_DOWN_FORT'] = recall_score(yt_dn, yp_dn, zero_division=0)
        metrics['F1_DOWN_FORT'] = f1_score(yt_dn, yp_dn, zero_division=0)
        if y_prob.ndim == 2 and y_prob.shape[1] >= 4:
            prob_down_fort_cond = y_prob[dn_idx, 0] / np.clip(y_prob[dn_idx, 0] + y_prob[dn_idx, 1], 1e-8, None)
            metrics['AUC_DOWN_FORT'] = safe_auc_binary(yt_dn, prob_down_fort_cond)

    if label:
        print(f"  [{label}] Acc_dir={metrics['Acc_dir']:.4f} | Precision_dir={metrics['Precision_dir_macro']:.4f} | Recall_dir={metrics['Recall_dir_macro']:.4f} | F1_dir={metrics['F1_dir']:.4f} | AUC_dir={metrics['AUC_dir'] if np.isfinite(metrics['AUC_dir']) else np.nan:.4f}")
    return metrics


def evaluate_model(model, loader, label=''):
    model.eval()
    all_preds, all_probs, all_targets = [], [], []
    with torch.no_grad():
        for bx, by in loader:
            logits = model(bx.to(device))
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = logits.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_probs.extend(probs)
            all_targets.extend(by.numpy())
    y_true = np.array(all_targets).astype(int)
    y_pred = np.array(all_preds).astype(int)
    y_prob = np.array(all_probs)
    metrics = compute_metrics_from_arrays(y_true, y_pred, y_prob, label=label)
    return metrics, y_pred, y_prob, y_true


## 10. Temperature Scaling — Calibration des probabilités

Paramètre $T$ appris sur le **val set** uniquement. $T>1$ → moins confiant (réduit l'overconfidence).


In [ ]:
class TemperatureScaler(nn.Module):
    """
    Module de calibration par Temperature Scaling.
    Enveloppe un modèle PyTorch existant et apprend le paramètre T.

    Usage :
        scaler = TemperatureScaler(trained_model)
        scaler.calibrate(val_loader, device)
        probs = scaler.predict_proba(X_tensor)
    """
    def __init__(self, model):
        super().__init__()
        self.model = model
        # T initialisé à 1.0 (pas de calibration) — appris sur le val set
        self.temperature = nn.Parameter(torch.ones(1) * 1.0)

    def forward(self, x):
        logits = self.model(x)
        return logits / self.temperature

    def calibrate(self, val_loader, device, lr=CONFIG['temperature_lr'],
                  epochs=CONFIG['temperature_epochs']):
        """Apprend T en minimisant la NLL sur le val set."""
        self.to(device)
        optimizer = torch.optim.LBFGS([self.temperature], lr=lr, max_iter=epochs)
        nll_criterion = nn.CrossEntropyLoss()

        # Collecter tous les logits et targets du val set (une seule fois)
        all_logits, all_targets = [], []
        self.model.eval()
        with torch.no_grad():
            for bx, by in val_loader:
                all_logits.append(self.model(bx.to(device)))
                all_targets.append(by.to(device))
        all_logits  = torch.cat(all_logits)
        all_targets = torch.cat(all_targets)

        def eval_fn():
            optimizer.zero_grad()
            loss = nll_criterion(all_logits / self.temperature, all_targets)
            loss.backward()
            return loss

        optimizer.step(eval_fn)
        print(f"  [TemperatureScaling] T optimal = {self.temperature.item():.4f}")

    @torch.no_grad()
    def predict_proba(self, x_tensor):
        self.eval()
        logits = self.forward(x_tensor.to(next(self.parameters()).device))
        return torch.softmax(logits, dim=1).cpu().numpy()


def plot_reliability_diagram(y_true, y_prob_uncal, y_prob_cal, n_bins=10, label=''):
    """
    Reliability diagram avant/après calibration.
    Chaque bin = observations dont la confiance du modèle est dans [b, b+1/n_bins).
    La courbe idéale est la diagonale (confiance = précision réelle).
    L'ECE (Expected Calibration Error) mesure l'aire entre la courbe et la diagonale.
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, probs, title in zip(axes,
                                  [y_prob_uncal, y_prob_cal],
                                  ['Avant calibration', 'Après Temperature Scaling']):
        # Classe la plus probable
        conf = probs.max(axis=1)
        correct = (probs.argmax(axis=1) == y_true).astype(float)

        bins = np.linspace(0, 1, n_bins + 1)
        bin_confs, bin_accs, bin_sizes = [], [], []
        for i in range(n_bins):
            mask = (conf >= bins[i]) & (conf < bins[i+1])
            if mask.sum() > 0:
                bin_confs.append(conf[mask].mean())
                bin_accs.append(correct[mask].mean())
                bin_sizes.append(mask.sum())

        ece = sum(s * abs(c - a) for s,c,a in zip(bin_sizes,bin_confs,bin_accs)) / len(y_true)

        ax.bar(bin_confs, bin_accs, width=0.08, alpha=0.7, color='steelblue', label='Modèle')
        ax.plot([0,1],[0,1],'r--', label='Calibration parfaite')
        ax.set_xlabel('Confiance'); ax.set_ylabel('Précision réelle')
        ax.set_title(f'{title}\nECE = {ece:.4f} {label}')
        ax.legend()
    plt.tight_layout(); plt.show()
    return ece


## 11. Stacking DL+ML + Threshold Optimization

**Stacking** : XGBoost méta-modèle sur les 7×4 probabilités OOF des modèles DL.
**Threshold** : seuil τ* optimal par régime sur le val set pour maximiser F1_dir.


In [ ]:
class MetaStackingClassifier:
    """
    Méta-classifieur par stacking.

    Niveau 1 : dict de modèles DL PyTorch (déjà entraînés)
    Niveau 2 : XGBoost entraîné sur les probabilités OOF du niveau 1

    La stratégie OOF (Out-of-Fold) garantit l'absence de leakage :
    les features du méta-modèle sont générées par les modèles niveau 1
    sur des données qu'ils n'ont pas vues pendant leur entraînement.
    """
    def __init__(self, base_models: dict, meta_model=None, n_folds=5):
        self.base_models = base_models  # {nom: modèle PyTorch}
        self.meta_model  = meta_model or XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric='mlogloss', random_state=SEED, n_jobs=-1
        )
        self.n_folds = n_folds
        self.fitted  = False
        self.thresholds_by_regime = {}

    def _get_proba(self, model, loader, device):
        """Extrait les probabilités softmax d'un modèle PyTorch sur un DataLoader."""
        model.eval()
        probs, targets = [], []
        with torch.no_grad():
            for bx, by in loader:
                logits = model(bx.to(device))
                probs.append(torch.softmax(logits, dim=1).cpu().numpy())
                targets.append(by.numpy())
        return np.vstack(probs), np.concatenate(targets)

    def generate_oof_features(self, X_train_seq, y_train, device):
        """
        Génère les features OOF pour le méta-modèle.
        Pour chaque fold temporel, les modèles sont évalués sur le fold
        qu'ils n'ont pas vu — garantit l'absence de leakage.
        """
        N = len(X_train_seq)
        n_models = len(self.base_models)
        oof_probs = np.zeros((N, n_models * 4))  # 4 classes par modèle

        tscv = TimeSeriesSplit(n_splits=self.n_folds)
        for fold_idx, (tr_idx, val_idx) in enumerate(tscv.split(X_train_seq)):
            if len(val_idx) < 10: continue
            x_val = torch.tensor(X_train_seq[val_idx], dtype=torch.float32)
            if x_val.dim() == 2:
                x_val = x_val.unsqueeze(1).repeat(1, CONFIG['lookback'], 1)
            y_val = torch.tensor(y_train[val_idx], dtype=torch.long)
            ds_val = TensorDataset(x_val, y_val)
            dl_val = DataLoader(ds_val, batch_size=256)

            for m_idx, (name, model) in enumerate(self.base_models.items()):
                probs, _ = self._get_proba(model, dl_val, device)
                oof_probs[val_idx, m_idx*4:(m_idx+1)*4] = probs

            if (fold_idx + 1) % 2 == 0:
                print(f"  [Stacking OOF] Fold {fold_idx+1}/{self.n_folds} ✓")

        return oof_probs

    def fit(self, X_train_seq, y_train, device):
        print("  [Stacking] Génération features OOF...")
        oof_features = self.generate_oof_features(X_train_seq, y_train, device)
        print(f"  [Stacking] Entraînement méta-modèle (XGBoost) sur {oof_features.shape}...")
        self.meta_model.fit(oof_features, y_train)
        self.fitted = True
        print("  [Stacking] Fit terminé.")

    def predict_proba(self, X_test_seq, device):
        """
        Test : chaque modèle de niveau 1 prédit sur tout le test,
        les probabilités sont concatenées et passées au méta-modèle.
        """
        assert self.fitted
        test_probs_list = []
        for name, model in self.base_models.items():
            x_t = torch.tensor(X_test_seq, dtype=torch.float32)
            if x_t.dim() == 2:
                x_t = x_t.unsqueeze(1).repeat(1, CONFIG['lookback'], 1)
            ds  = TensorDataset(x_t, torch.zeros(len(x_t), dtype=torch.long))
            dl  = DataLoader(ds, batch_size=256)
            probs, _ = self._get_proba(model, dl, device)
            test_probs_list.append(probs)
        meta_features = np.hstack(test_probs_list)
        return self.meta_model.predict_proba(meta_features)

    def predict(self, X_test_seq, device):
        return self.predict_proba(X_test_seq, device).argmax(axis=1)


def optimize_direction_threshold(probs, y_true, regime_mask=None, n_steps=50):
    """
    Optimise le seuil de décision UP/DOWN sur le val set pour maximiser F1_dir.

    probs      : (N, 4) probabilités softmax
    y_true     : (N,) labels 0-3
    regime_mask: masque optionnel pour optimiser par régime

    Retourne le seuil optimal τ* ∈ [0.3, 0.7]
    """
    dir_map   = {0:'DOWN',1:'DOWN',2:'UP',3:'UP'}
    # P(UP) = sum des probabilités des classes UP
    p_up      = probs[:, 2] + probs[:, 3]
    yd_true   = [dir_map[y] for y in y_true]

    thresholds = np.linspace(0.3, 0.7, n_steps)
    best_thr, best_f1 = 0.5, -1

    for thr in thresholds:
        yd_pred = ['UP' if p > thr else 'DOWN' for p in p_up]
        if regime_mask is not None:
            yd_true_r = [yd_true[i] for i in range(len(yd_true)) if regime_mask[i]]
            yd_pred_r = [yd_pred[i] for i in range(len(yd_pred)) if regime_mask[i]]
        else:
            yd_true_r, yd_pred_r = yd_true, yd_pred
        f1 = f1_score(yd_true_r, yd_pred_r, average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr

    return best_thr, best_f1


## 12. Options Flow et Corrélation Implicite

In [ ]:
def build_options_flow_features(df_raw, train_end_idx):
    """
    Construit des proxies du flux d'options depuis les données disponibles.

    PCR direct : téléchargé depuis CBOE via FRED (PUTCALLRATIO)
    ou estimé depuis les volumes VIX/VVIX si non disponible.
    """
    feats = pd.DataFrame(index=df_raw.index)
    t0 = time.time()

    # ── Tentative de téléchargement du PCR CBOE via FRED ─────────────────────
    # PUTCALLRATIO : ratio Put/Call total, disponible sur FRED
    try:
        pcr = web.DataReader('PUTCALLRATIO', 'fred',
                              start=CONFIG['start_date']).squeeze()
        pcr = pcr.reindex(df_raw.index, method='ffill')
        feats['pcr_total']    = pcr
        feats['pcr_zscore']   = (pcr - pcr.iloc[:train_end_idx].mean()) / pcr.iloc[:train_end_idx].std()
        feats['pcr_ma5']      = pcr.rolling(5, min_periods=3).mean()
        feats['pcr_spike']    = (pcr - feats['pcr_ma5']) / feats['pcr_ma5'].replace(0, np.nan)
        print(f"  [PCR] CBOE Put/Call Ratio chargé depuis FRED ({time.time()-t0:.1f}s)")
    except Exception as e:
        print(f"  [PCR] FRED non disponible ({e}) — proxy via VIX/VVIX")
        # Proxy : ratio VVIX/VIX comme indicateur de demande d'options
        vix_cols  = [c for c in df_raw.columns if 'VIX' in c and 'VVIX' not in c]
        vvix_cols = [c for c in df_raw.columns if 'VVIX' in c]
        if vix_cols and vvix_cols:
            vix  = df_raw[vix_cols[0]].ffill()
            vvix = df_raw[vvix_cols[0]].ffill()
            pcr_proxy = vvix / vix.replace(0, np.nan)
            feats['pcr_proxy']       = pcr_proxy
            feats['pcr_proxy_zscore']= (pcr_proxy - pcr_proxy.iloc[:train_end_idx].mean()) /                                         pcr_proxy.iloc[:train_end_idx].std()

    # ── Corrélation implicite (proxy via secteurs ETF) ────────────────────────
    # σ_indice = VIX/100 (volatilité implicite SPX annualisée)
    # σ_secteur = vol réalisée rolling 21j des ETFs sectoriels
    # Poids égaux (simplification — un vrai calcul utiliserait les poids de l'indice)
    sector_etfs = [c for c in df_raw.columns
                   if any(s in c for s in ['XLK','XLF','XLE','XLV','XLU','XLB','XLI','XLY'])]
    vix_col = [c for c in df_raw.columns if 'IDX_VIX' in c or (c.endswith('VIX') and 'VXN' not in c and 'VVIX' not in c)]

    if sector_etfs and vix_col:
        vix  = df_raw[vix_col[0]].ffill() / 100  # en décimal
        w    = 1.0 / len(sector_etfs)

        # Volatilités réalisées des secteurs (rolling 21j)
        sector_vols = {}
        for col in sector_etfs:
            ret = np.log(df_raw[col].ffill() / df_raw[col].ffill().shift(1))
            sector_vols[col] = ret.rolling(21, min_periods=10).std() * np.sqrt(252)

        # Numérateur : σ_indice² - Σ wᵢ²σᵢ²
        sum_wi2_sigma2 = sum(w**2 * sv**2 for sv in sector_vols.values())
        # Dénominateur : Σᵢ≠ⱼ wᵢwⱼσᵢσⱼ (approximé par σ_indice² - sum_wi2_sigma2)
        numerator = vix**2 - sum_wi2_sigma2
        # Dénominateur : somme des covariances croisées attendues
        n = len(sector_etfs)
        avg_sigma  = pd.concat(sector_vols.values(), axis=1).mean(axis=1)
        denominator = (n**2 - n) * w**2 * avg_sigma**2

        impl_corr = (numerator / denominator.replace(0, np.nan)).clip(-1, 1)
        impl_corr_smooth = impl_corr.rolling(5, min_periods=3).mean()

        feats['impl_corr']        = impl_corr_smooth
        feats['impl_corr_zscore'] = (impl_corr_smooth - impl_corr_smooth.iloc[:train_end_idx].mean()) /                                      impl_corr_smooth.iloc[:train_end_idx].std()
        feats['impl_corr_delta']  = impl_corr_smooth.diff()
        print(f"  [Impl Corr] Corrélation implicite calculée sur {len(sector_etfs)} secteurs ({time.time()-t0:.1f}s)")

    return feats.replace([np.inf, -np.inf], np.nan)


## 13. Conformal Prediction — Couverture garantie à $1-\alpha$

Non-conformity score : $s_i = 1 - P(y_i|x_i)$. Quantile $\hat{q}$ sur cal set.
Ensemble de confiance : $C(x) = \{k : 1-P(k|x) \leq \hat{q}\}$.


In [ ]:
class TemporalConformalClassifier:
    """
    Conformal Prediction adapté aux séries temporelles.

    Différence vs conformal classique : en finance, les observations ne sont pas
    échangeables (il y a une dépendance temporelle). On utilise une variante
    où le cal set est **chronologiquement postérieur** au train, pour respecter
    la causalité temporelle.

    Référence :
    - Vovk et al. (1999) — Conformal Prediction
    - Barber et al. (2023) — Conformal Prediction for Time Series
    """
    def __init__(self, alpha=CONFIG['conformal_alpha']):
        self.alpha = alpha
        self.q_hat = None  # quantile de calibration

    def calibrate(self, model, cal_loader, device):
        """
        Calcule les non-conformity scores sur l'ensemble de calibration.

        score_i = 1 - P(y_i | x_i)  (plus le score est élevé, plus la prédiction est 'surprenante')
        q_hat = quantile (1-alpha) des scores → seuil de l'ensemble de confiance
        """
        model.eval()
        scores = []
        with torch.no_grad():
            for bx, by in cal_loader:
                logits = model(bx.to(device))
                probs  = torch.softmax(logits, dim=1).cpu().numpy()
                for i, label in enumerate(by.numpy()):
                    # Non-conformity score : 1 - probabilité de la vraie classe
                    scores.append(1.0 - probs[i, label])

        scores   = np.array(scores)
        n        = len(scores)
        # Quantile ajusté pour la garantie de couverture finie
        q_level  = np.ceil((n + 1) * (1 - self.alpha)) / n
        self.q_hat = float(np.quantile(scores, min(q_level, 1.0)))
        print(f"  [Conformal] q_hat = {self.q_hat:.4f} (alpha={self.alpha}, n_cal={n})")

    def predict_set(self, model, x_tensor, device):
        """
        Prédit l'ensemble de confiance C(x) pour chaque observation.
        C(x) = {k : 1 - P(k|x) <= q_hat}

        Retourne une liste de listes : chaque sous-liste contient les classes
        plausibles pour l'observation correspondante.
        """
        assert self.q_hat is not None, "Calibrer d'abord avec .calibrate()"
        model.eval()
        with torch.no_grad():
            logits = model(x_tensor.to(device))
            probs  = torch.softmax(logits, dim=1).cpu().numpy()

        prediction_sets = []
        for i in range(len(probs)):
            # Inclure toutes les classes dont le non-conformity score <= q_hat
            conf_set = [k for k in range(4) if (1 - probs[i, k]) <= self.q_hat]
            prediction_sets.append(conf_set)

        return prediction_sets

    def evaluate_coverage(self, model, test_loader, device):
        """
        Vérifie empiriquement que la couverture est bien >= 1-alpha.
        Corollaire : si coverage < 1-alpha, il y a un problème (données non-échangeables
        ou calibration insuffisante).
        """
        covered = 0
        total   = 0
        set_sizes = []

        model.eval()
        with torch.no_grad():
            for bx, by in test_loader:
                x_t   = bx.to(device)
                logits = model(x_t)
                probs  = torch.softmax(logits, dim=1).cpu().numpy()
                labels = by.numpy()

                for i, label in enumerate(labels):
                    conf_set = [k for k in range(4) if (1 - probs[i, k]) <= self.q_hat]
                    if label in conf_set:
                        covered += 1
                    set_sizes.append(len(conf_set))
                    total += 1

        coverage   = covered / total
        avg_set_sz = np.mean(set_sizes)
        print(f"  [Conformal] Couverture empirique : {coverage:.4f} (cible : {1-self.alpha:.2f})")
        print(f"  [Conformal] Taille moyenne des ensembles : {avg_set_sz:.2f} / 4 classes")
        return coverage, avg_set_sz

    def trading_signal(self, prediction_set):
        """
        Convertit l'ensemble de confiance en signal de trading.
        Plus l'ensemble est petit et concentré sur des classes extrêmes,
        plus le signal est fort.
        """
        CLASS_LABELS = ['DOWN_FORT','DOWN_FAIBLE','UP_FAIBLE','UP_FORT']
        set_labels = [CLASS_LABELS[k] for k in prediction_set]
        n = len(prediction_set)

        if n == 1:
            return {'signal': set_labels[0], 'confidence': 'FORTE', 'sizing': 1.0}
        elif n == 2:
            if all(k >= 2 for k in prediction_set):
                return {'signal': 'UP',   'confidence': 'MODÉRÉE', 'sizing': 0.5}
            elif all(k < 2  for k in prediction_set):
                return {'signal': 'DOWN', 'confidence': 'MODÉRÉE', 'sizing': 0.5}
        elif n >= 3:
            return {'signal': 'INCERTAIN', 'confidence': 'FAIBLE', 'sizing': 0.0}
        return {'signal': 'NEUTRE', 'confidence': 'NULLE', 'sizing': 0.0}


## 14. Adversarial Validation — Détection du drift

In [ ]:
def adversarial_validation(X_train, X_test, feature_names=None,
                            threshold=CONFIG['adv_val_threshold'],
                            n_estimators=100):
    """
    Validation adversariale : détecte le drift de distribution entre train et test.

    Algorithme :
    1. Créer dataset binaire {train:0, test:1}
    2. Entraîner RandomForest pour distinguer les deux
    3. AUC proche de 0.5 = pas de drift / proche de 1.0 = drift sévère

    Paramètres
    ----------
    X_train : array (N_train, F) — features du train
    X_test  : array (N_test, F)  — features du test
    threshold: AUC au-dessus duquel on signale un drift

    Retourne
    --------
    auc         : float — AUC du classifieur adversarial
    drift_feats : list — features les plus responsables du drift (SHAP)
    """
    t0 = time.time()

    # Sous-échantillonner pour équilibrer les classes
    n_min = min(len(X_train), len(X_test))
    idx_tr = np.random.choice(len(X_train), n_min, replace=False)
    idx_te = np.random.choice(len(X_test),  n_min, replace=False)

    X_adv = np.vstack([X_train[idx_tr], X_test[idx_te]])
    y_adv = np.array([0]*n_min + [1]*n_min)

    # Validation croisée temporelle (pas de shuffle — respecter la causalité)
    tscv  = TimeSeriesSplit(n_splits=5)
    aucs  = []
    clf   = RandomForestClassifier(n_estimators=n_estimators, max_depth=5,
                                    random_state=SEED, n_jobs=-1)

    for tr_idx, val_idx in tscv.split(X_adv):
        clf.fit(X_adv[tr_idx], y_adv[tr_idx])
        probs = clf.predict_proba(X_adv[val_idx])
        if probs.shape[1] < 2:
            aucs.append(0.5)
            continue
        probs = probs[:, 1]
        auc   = roc_auc_score(y_adv[val_idx], probs)
        aucs.append(auc)

    mean_auc = np.mean(aucs)

    print(f"\n  [Adversarial Validation] AUC = {mean_auc:.4f} ({time.time()-t0:.1f}s)")
    if mean_auc > threshold:
        print(f"  [ALERTE] Drift significatif détecté (AUC > {threshold})")
        print(f"  Action suggérée : restreindre la fenêtre de train aux données récentes")
    else:
        print(f"  [OK] Pas de drift significatif (AUC <= {threshold})")

    # Identifier les features responsables du drift
    drift_feats = []
    if feature_names is not None:
        # Entraîner sur tout le dataset pour l'analyse SHAP
        clf.fit(X_adv, y_adv)
        importances = pd.Series(clf.feature_importances_, index=feature_names)
        top_drift = importances.nlargest(10)
        drift_feats = top_drift.index.tolist()

        print(f"  Top features responsables du drift :")
        for feat, imp in top_drift.items():
            print(f"    {feat:<45} importance = {imp:.4f}")

    return mean_auc, drift_feats


def plot_distribution_shift(X_train, X_test, feature_names, n_features=6):
    """
    Visualise le shift de distribution pour les N features les plus importantes.
    Utile pour comprendre concrètement comment la distribution a changé.
    """
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    for i, feat in enumerate(feature_names[:n_features]):
        if feat not in [feature_names[j] for j in range(len(feature_names))]:
            continue
        feat_idx = list(feature_names).index(feat)
        ax = axes[i]
        ax.hist(X_train[:, feat_idx], bins=30, alpha=0.6, label='Train', color='blue', density=True)
        ax.hist(X_test[:,  feat_idx], bins=30, alpha=0.6, label='Test',  color='red',  density=True)
        ax.set_title(f'{feat[:30]}')
        ax.legend()

    plt.suptitle('Distribution Shift : Train vs Test', fontsize=14)
    plt.tight_layout()
    plt.show()


## 15. Stress Testing sectoriel — 5 crises historiques

In [ ]:
def stress_test_models(models_dict, df_full, feature_cols, target_col,
                       scaler, train_end_date, lookback=CONFIG['lookback']):
    """
    Évalue chaque modèle sur les grandes périodes de crise historiques.

    Paramètres
    ----------
    models_dict  : dict {nom: modèle PyTorch calibré}
    df_full      : DataFrame complet (features + target)
    feature_cols : colonnes de features
    train_end_date : date de fin du train (pour s'assurer qu'on ne teste que sur le test)
    """
    results = {}

    print("\n" + "="*60)
    print("STRESS TESTING — Performance sur les grandes crises")
    print("="*60)

    for crisis_name, (start, end) in CRISIS_PERIODS.items():
        # Vérifier que la crise est dans le test set
        crisis_start = pd.Timestamp(start)
        crisis_end   = pd.Timestamp(end)
        train_end    = pd.Timestamp(train_end_date)

        if crisis_end <= train_end:
            print(f"  [SKIP] {crisis_name} : antérieure à la fin du train")
            continue

        # Données de crise (subset du test)
        crisis_mask = ((df_full.index >= crisis_start) &
                        (df_full.index <= crisis_end) &
                        (df_full.index > train_end))
        df_crisis = df_full.loc[crisis_mask].dropna(subset=[target_col])

        if len(df_crisis) < 10:
            print(f"  [SKIP] {crisis_name} : moins de 10 observations")
            continue

        y_crisis = df_crisis[target_col].values.astype(int)
        X_crisis = scaler.transform(df_crisis[feature_cols].fillna(0))

        # Créer les séquences lookback
        if len(X_crisis) <= lookback:
            print(f"  [SKIP] {crisis_name} : trop peu d'observations pour le lookback")
            continue

        # Construire le tenseur de séquences
        X_seq = np.array([X_crisis[i:i+lookback] for i in range(len(X_crisis)-lookback)])
        y_seq = y_crisis[lookback:]
        x_t   = torch.tensor(X_seq, dtype=torch.float32)

        results[crisis_name] = {'n_obs': len(y_seq)}
        print(f"\n  📉 {crisis_name} ({start} → {end}) — {len(y_seq)} obs")

        for model_name, model in models_dict.items():
            model.eval()
            with torch.no_grad():
                logits = model(x_t.to(device))
                preds  = logits.argmax(1).cpu().numpy()

            # Métriques hiérarchiques
            dir_map  = {0:'DOWN',1:'DOWN',2:'UP',3:'UP'}
            yd_true  = [dir_map[y] for y in y_seq]
            yd_pred  = [dir_map[p] for p in preds]
            f1_dir   = f1_score(yd_true, yd_pred, average='macro', zero_division=0)
            acc_dir  = accuracy_score(yd_true, yd_pred)

            up_idx = [i for i,y in enumerate(y_seq) if dir_map[y]=='UP']
            f1_uf  = 0.0
            if up_idx:
                yt_up = ['FORT' if y_seq[i]==3 else 'FAIBLE' for i in up_idx]
                yp_up = ['FORT' if preds[i]==3 else 'FAIBLE' for i in up_idx]
                f1_uf = f1_score(yt_up,yp_up,pos_label='FORT',average='binary',zero_division=0)

            results[crisis_name][model_name] = {
                'f1_dir': f1_dir, 'acc_dir': acc_dir, 'f1_up_fort': f1_uf
            }
            print(f"    {model_name:<20} F1_dir={f1_dir:.3f}  Acc={acc_dir:.3f}  F1_UP_FORT={f1_uf:.3f}")

    # Résumé comparatif
    print("\n" + "="*60)
    print("RÉSUMÉ STRESS TEST — F1_dir par crise et modèle")
    print("="*60)
    crisis_df_rows = []
    for crisis, data in results.items():
        for model_name in models_dict:
            if model_name in data:
                crisis_df_rows.append({
                    'Crisis': crisis, 'Model': model_name,
                    'F1_dir': data[model_name]['f1_dir'],
                    'F1_UP_FORT': data[model_name]['f1_up_fort'],
                    'N_obs': data['n_obs']
                })
    if crisis_df_rows:
        df_stress = pd.DataFrame(crisis_df_rows)
        pivot = df_stress.pivot(index='Crisis', columns='Model', values='F1_dir')
        print(pivot.round(3).to_string())

    return results


## 16. Ensemble Asymétrique — Pondération par régime

In [ ]:
class AsymmetricEnsemble:
    """
    Ensemble dont les poids varient selon le régime de marché courant.

    Pour chaque régime (CALM/NORMAL/STRESS), apprend un vecteur de poids
    qui maximise le F1_dir sur le val set filtré par ce régime.

    La sélection du régime est basée sur le niveau de VIX :
    - VIX < REGIME_THRESHOLDS['calm']   → CALM
    - VIX >= REGIME_THRESHOLDS['stress'] → STRESS
    - Sinon                               → NORMAL
    """
    def __init__(self, models_dict: dict):
        self.models = models_dict
        self.weights_by_regime = {
            'CALM':   np.ones(len(models_dict)) / len(models_dict),  # uniform init
            'NORMAL': np.ones(len(models_dict)) / len(models_dict),
            'STRESS': np.ones(len(models_dict)) / len(models_dict),
        }
        self.fitted = False

    def _get_vix_regime(self, vix_value):
        if vix_value < REGIME_THRESHOLDS['calm']:
            return 'CALM'
        elif vix_value >= REGIME_THRESHOLDS['stress']:
            return 'STRESS'
        return 'NORMAL'

    def _get_all_proba(self, loader, device):
        """Collecte les probabilités de tous les modèles sur un DataLoader."""
        all_probs = {name: [] for name in self.models}
        all_targets = []

        for bx, by in loader:
            bx = bx.to(device)
            for name, model in self.models.items():
                model.eval()
                with torch.no_grad():
                    probs = torch.softmax(model(bx), dim=1).cpu().numpy()
                all_probs[name].append(probs)
            all_targets.extend(by.numpy())

        return {k: np.vstack(v) for k, v in all_probs.items()}, np.array(all_targets)

    def fit_regime_weights(self, val_loader, vix_val_series, device):
        """
        Apprend les poids optimaux par régime sur le val set.
        Minimise la NLL (ou maximise F1_dir) par régime.
        """
        print("  [Ensemble Asymétrique] Optimisation des poids par régime...")
        all_probs_dict, y_val = self._get_all_proba(val_loader, device)
        dir_map = {0:'DOWN',1:'DOWN',2:'UP',3:'UP'}

        for regime in ['CALM', 'NORMAL', 'STRESS']:
            # Identifier les observations de ce régime
            if vix_val_series is not None and len(vix_val_series) == len(y_val):
                regime_mask = np.array([
                    self._get_vix_regime(v) == regime
                    for v in vix_val_series.values
                ])
            else:
                # Fallback : régime par tertile de l'index
                regime_mask = np.ones(len(y_val), dtype=bool)

            if regime_mask.sum() < 10:
                print(f"    {regime}: trop peu d'obs ({regime_mask.sum()}), poids uniformes")
                continue

            y_reg = y_val[regime_mask]
            yd_true_reg = [dir_map[y] for y in y_reg]

            # Optimisation par recherche sur grille simple (Dirichlet sampling)
            best_weights = np.ones(len(self.models)) / len(self.models)
            best_f1 = -1

            # 200 combinaisons aléatoires de poids
            np.random.seed(SEED)
            for _ in range(200):
                # Tirer des poids via Dirichlet (distribués sur le simplex)
                w = np.random.dirichlet(np.ones(len(self.models)))
                # Ensemble pondéré
                ensemble_probs = sum(
                    w[i] * all_probs_dict[name][regime_mask]
                    for i, name in enumerate(self.models)
                )
                yd_pred = [dir_map[p] for p in ensemble_probs.argmax(axis=1)]
                f1 = f1_score(yd_true_reg, yd_pred, average='macro', zero_division=0)
                if f1 > best_f1:
                    best_f1, best_weights = f1, w

            self.weights_by_regime[regime] = best_weights
            print(f"    {regime}: F1_dir = {best_f1:.4f} | poids = {dict(zip(self.models.keys(), best_weights.round(3)))}")

        self.fitted = True

    def predict_proba(self, x_tensor, vix_value, device):
        """
        Prédit les probabilités avec les poids du régime courant.

        vix_value : niveau de VIX du jour de la prédiction
        """
        regime  = self._get_vix_regime(vix_value)
        weights = self.weights_by_regime[regime]

        ensemble_probs = np.zeros((len(x_tensor), 4))
        for i, (name, model) in enumerate(self.models.items()):
            model.eval()
            with torch.no_grad():
                probs = torch.softmax(model(x_tensor.to(device)), dim=1).cpu().numpy()
            ensemble_probs += weights[i] * probs

        return ensemble_probs, regime

    def evaluate(self, test_loader, vix_test_series, device):
        """Évalue l'ensemble asymétrique en utilisant le régime de chaque jour."""
        all_preds, all_targets = [], []
        dir_map = {0:'DOWN',1:'DOWN',2:'UP',3:'UP'}

        for batch_idx, (bx, by) in enumerate(test_loader):
            # VIX moyen du batch pour déterminer le régime
            batch_size = len(bx)
            start_idx  = batch_idx * test_loader.batch_size
            end_idx    = min(start_idx + batch_size, len(vix_test_series))
            if end_idx > start_idx and vix_test_series is not None:
                vix_val = vix_test_series.iloc[start_idx:end_idx].mean()
            else:
                vix_val = 20.0  # NORMAL par défaut

            probs, regime = self.predict_proba(bx, vix_val, device)
            all_preds.extend(probs.argmax(axis=1))
            all_targets.extend(by.numpy())

        yd_t = [dir_map[y] for y in all_targets]
        yd_p = [dir_map[p] for p in all_preds]
        f1   = f1_score(yd_t, yd_p, average='macro', zero_division=0)
        print(f"  [Ensemble Asymétrique] F1_dir = {f1:.4f}")
        return f1


## 17. SMOTE adaptatif

In [ ]:
def select_best_sampler(X_train, y_train):
    samplers = {
        'SMOTE':           SMOTE(random_state=SEED),
        'BorderlineSMOTE': BorderlineSMOTE(random_state=SEED, kind='borderline-1'),
        'SMOTETomek':      SMOTETomek(random_state=SEED),
    }
    pilot = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=SEED, n_jobs=-1)
    best_name, best_f1, best_s = 'SMOTE', -1, samplers['SMOTE']
    for name, samp in samplers.items():
        try:
            Xr, yr = samp.fit_resample(X_train, y_train)
            n = len(Xr); scores = []
            for fold in range(3):
                te = n*(fold+2)//4; ve = n*(fold+3)//4
                pilot.fit(Xr[:te], yr[:te])
                p = pilot.predict(Xr[te:ve])
                scores.append(f1_score(yr[te:ve], p, average='macro', zero_division=0))
            f = np.mean(scores)
            if f > best_f1: best_f1, best_name, best_s = f, name, samp
        except: pass
    print(f"  Sampler: {best_name} (F1={best_f1:.4f})")
    return best_name, best_s


## 18. Pipeline Final — Consolidé

**Corrections appliquées :**
- `PreScaledDataset` pour val/test (zéro double-scaling)
- Split 70/10/20 explicite : train / val (calibration) / test (évaluation)
- Adversarial validation avec guard `probs.shape[1] < 2`
- Stacking : tenseurs 3D garantis `(N, lookback, features)`
- `predict_proba` de `TemperatureScaler` sans argument `device`
- Toutes les crises antérieures à `val_end` skippées proprement


In [ ]:
def run_pipeline_final(df_raw, horizon=5):
    t_total = time.time()
    print(f"\n{'='*60}\nPIPELINE FINAL | h={horizon}j\n{'='*60}")

    # Identifier VIX et SPX
    vix_col = [c for c in df_raw.columns
               if ('IDX_VIX' in c or c.endswith('_VIX'))
               and 'VXN' not in c and 'VVIX' not in c][0]
    spx_cols = [c for c in df_raw.columns if 'GSPC' in c or c=='SPY']
    spx_col  = spx_cols[0] if spx_cols else None
    vix_series = df_raw[vix_col].ffill()

    # Split 70/10/20
    all_dates = df_raw.dropna(how='all').index.sort_values()
    n = len(all_dates)
    train_end  = all_dates[int(n*0.70)]
    val_end    = all_dates[int(n*0.80)]
    train_end_idx = int(n*0.70)
    print(f"  Train  → {train_end.date()} | Val → {val_end.date()} | Test → {all_dates[-1].date()}")

    # Features
    print(f"  [TS Features] ({time.time()-t_total:.1f}s)")
    ts_feats  = build_ts_features(df_raw, train_end_idx)
    spx_s     = df_raw[spx_col].ffill() if spx_col else pd.Series(1., index=df_raw.index)
    adv_feats = build_advanced_features(df_raw, vix_series, spx_s, train_end_idx)
    print(f"  [Options Flow] ({time.time()-t_total:.1f}s)")
    opt_feats = build_options_flow_features(df_raw, train_end_idx)

    # Rendements multi-horizons
    ret_list = []
    for col in df_raw.columns:
        for w in [1,5,21]:
            s = df_raw[col].pct_change(w); s.name = f'{col}_ret{w}d'
            ret_list.append(s)
    ret_df = pd.concat(ret_list, axis=1)

    df_all = pd.concat([df_raw, ts_feats, adv_feats, opt_feats, ret_df], axis=1)
    df_all = df_all.replace([np.inf,-np.inf], np.nan)

    # Cible
    target, regime, _, _ = build_amplitude_target(vix_series, horizon, train_end_idx)
    df_all = df_all.reindex(target.index); df_all[TARGET_COL] = target

    # Train / Val / Test DataFrames
    df_tr   = df_all.loc[df_all.index <= train_end].dropna(subset=[TARGET_COL])
    df_val  = df_all.loc[(df_all.index > train_end) & (df_all.index <= val_end)].dropna(subset=[TARGET_COL])
    df_te   = df_all.loc[df_all.index > val_end].dropna(subset=[TARGET_COL])

    feat_cols = [c for c in df_all.columns if c != TARGET_COL]
    y_tr   = df_tr[TARGET_COL].values.astype(int)
    y_val  = df_val[TARGET_COL].values.astype(int)
    y_te   = df_te[TARGET_COL].values.astype(int)

    print(f"  Train: {len(df_tr)} | Val: {len(df_val)} | Test: {len(df_te)}")

    # Interactions + SHAP
    print(f"  [Interactions] ({time.time()-t_total:.1f}s)")
    X_tr_base = df_tr[feat_cols].replace([np.inf,-np.inf],np.nan).fillna(0)
    idf_train, interaction_specs, _ = generate_interactions_all_features_batched(
        X_tr_base, y_tr, feature_cols=feat_cols,
        max_keep=CONFIG['max_interaction_keep'],
        batch_size=CONFIG['interaction_batch_size'],
        keep_per_batch=CONFIG['interaction_keep_per_batch'],
    )
    idf_val = apply_interaction_specs(df_val[feat_cols].replace([np.inf,-np.inf],np.nan).fillna(0),
                                       interaction_specs)
    idf_te  = apply_interaction_specs(df_te[feat_cols].replace([np.inf,-np.inf],np.nan).fillna(0),
                                       interaction_specs)

    def merge_ext(df_base, idf, feat_cols, inter_cols):
        base_clean = df_base[feat_cols].replace([np.inf,-np.inf],np.nan).fillna(0)
        for c in inter_cols:
            if c not in idf.columns: idf[c] = 0.
        return pd.concat([base_clean, idf[inter_cols]], axis=1)

    inter_cols = idf_train.columns.tolist()
    df_tr_ext  = merge_ext(df_tr,  idf_train, feat_cols, inter_cols)
    df_val_ext = merge_ext(df_val, idf_val,   feat_cols, inter_cols)
    df_te_ext  = merge_ext(df_te,  idf_te,    feat_cols, inter_cols)
    ext_cols   = df_tr_ext.columns.tolist()

    # Scaler fitté sur train uniquement
    sc = RobustScaler()
    X_tr_sc  = sc.fit_transform(df_tr_ext.fillna(0))
    X_val_sc = sc.transform(df_val_ext.fillna(0))
    X_te_sc  = sc.transform(df_te_ext.fillna(0))

    # SHAP sélection finale
    print(f"  [SHAP] ({time.time()-t_total:.1f}s)")
    X_tr_df  = pd.DataFrame(X_tr_sc, columns=ext_cols, index=df_tr.index)
    top_final, _ = shap_select_features(X_tr_df, y_tr, CONFIG['top_n_final'], 'final')
    feat_idx  = [ext_cols.index(f) for f in top_final]
    X_tr_f   = X_tr_sc[:, feat_idx]
    X_val_f  = X_val_sc[:, feat_idx]
    X_te_f   = X_te_sc[:,  feat_idx]
    print(f"  Features finales: {len(top_final)} ({time.time()-t_total:.1f}s)")

    # Adversarial Validation
    print(f"  [Adversarial Validation] ({time.time()-t_total:.1f}s)")
    adv_auc, drift_feats = adversarial_validation(X_tr_f, X_te_f, feature_names=top_final)

    # SMOTE sur le train
    sampler_name, best_samp = select_best_sampler(X_tr_f, y_tr)
    X_res, y_res = best_samp.fit_resample(X_tr_f, y_tr)

    # Datasets — PreScaledDataset pour val/test (zéro double-scaling)
    sc_seq = RobustScaler().fit(X_res)   # scaler final sur train SMOTE
    X_res_sc = sc_seq.transform(X_res)
    X_val_sc2 = sc_seq.transform(X_val_f)
    X_te_sc2  = sc_seq.transform(X_te_f)

    val_split = int(len(X_res_sc)*0.85)
    ds_tr  = PreScaledDataset(X_res_sc[:val_split], y_res[:val_split])
    ds_va  = PreScaledDataset(X_res_sc[val_split:], y_res[val_split:])
    ds_cal = PreScaledDataset(X_val_sc2, y_val)   # calibration Temperature Scaling
    ds_te  = PreScaledDataset(X_te_sc2,  y_te)

    dl_tr  = DataLoader(ds_tr,  batch_size=CONFIG['batch_size'], shuffle=True)
    dl_va  = DataLoader(ds_va,  batch_size=256)
    dl_cal = DataLoader(ds_cal, batch_size=256)
    dl_te  = DataLoader(ds_te,  batch_size=256)

    input_dim    = len(top_final)
    class_weights = compute_class_weights(y_res)

    # Entraînement 7 modèles
    models_def = {
        'LSTM':        VIX_LSTM(input_dim),
        'TCN':         VIX_TCN(input_dim),
        'Transformer': VIX_Transformer(input_dim),
        'CNN-LSTM':    VIX_CNNLSTM(input_dim),
        'N-BEATS':     VIX_NBeats(input_dim, CONFIG['lookback']),
        'TFT':         VIX_TFT(input_dim),
        'Mamba':       VIX_Mamba(input_dim),
    }
    trained, base_res, predictions = {}, {}, {}
    for name, model in models_def.items():
        print(f"\n  ── {name} ({time.time()-t_total:.1f}s) ──")
        model = model.to(device)
        train_model(model, dl_tr, dl_va, class_weights=class_weights, label=name)
        met, preds, probs, y_true_e = evaluate_model(model, dl_te, label=name)
        trained[name]     = model
        base_res[name]    = met
        predictions[name] = {'y_true':y_true_e,'y_pred':preds,'y_prob':probs}

    # Ensemble pondéré par F1_dir
    model_names = list(trained.keys())
    w_arr = np.array([base_res[n].get('F1_dir',0.) for n in model_names])
    w_arr = np.clip(w_arr - w_arr.min() + 1e-6, 0, None)
    w_arr = w_arr / w_arr.sum()
    min_len = min(len(predictions[n]['y_prob']) for n in model_names)
    ens_probs = np.sum([predictions[n]['y_prob'][:min_len]*w for n,w in zip(model_names,w_arr)], axis=0)
    ens_preds = ens_probs.argmax(1)
    y_true_ens = predictions[model_names[0]]['y_true'][:min_len]
    ens_met = compute_metrics_from_arrays(y_true_ens, ens_preds, ens_probs, label='ENSEMBLE')
    base_res['ENSEMBLE'] = ens_met

    # Temperature Scaling
    print(f"\n  [Temperature Scaling] ({time.time()-t_total:.1f}s)")
    calibrated = {}
    for name, model in trained.items():
        ts = TemperatureScaler(model).to(device)
        ts.calibrate(dl_cal, device)
        calibrated[name] = ts

    # Stacking — tenseurs 3D garantis
    print(f"\n  [Stacking] ({time.time()-t_total:.1f}s)")
    stacker = MetaStackingClassifier(trained)
    stacker.fit(X_res_sc, y_res, device)
    stack_pr  = stacker.predict_proba(X_te_sc2, device)
    dm = {0:'DOWN',1:'DOWN',2:'UP',3:'UP'}
    stack_f1  = f1_score([dm[y] for y in y_te[:len(stack_pr)]],
                          [dm[p] for p in stack_pr.argmax(1)],
                          average='macro', zero_division=0)
    print(f"  Stacking F1_dir = {stack_f1:.4f}")

    # Threshold Optimization
    print(f"\n  [Threshold Opt.] ({time.time()-t_total:.1f}s)")
    thresholds = {}
    for name, cal in calibrated.items():
        xv = torch.tensor(X_val_sc2, dtype=torch.float32)
        xv = xv.unsqueeze(1).repeat(1, CONFIG['lookback'], 1)
        pv = cal.predict_proba(xv)
        thr, f1t = optimize_direction_threshold(pv, y_val)
        thresholds[name] = thr
        print(f"    {name}: τ*={thr:.3f} → F1={f1t:.4f}")

    # Conformal Prediction
    print(f"\n  [Conformal] ({time.time()-t_total:.1f}s)")
    best_name = max(base_res, key=lambda x: base_res[x].get('F1_dir',0.) if x!='ENSEMBLE' else 0.)
    conformal = TemporalConformalClassifier(alpha=CONFIG['conformal_alpha'])
    conformal.calibrate(trained[best_name], dl_cal, device)
    coverage, avg_sz = conformal.evaluate_coverage(trained[best_name], dl_te, device)

    # Ensemble Asymétrique
    print(f"\n  [Ensemble Asymétrique] ({time.time()-t_total:.1f}s)")
    vix_val_s  = vix_series.reindex(df_val.index)
    vix_test_s = vix_series.reindex(df_te.index)
    asym = AsymmetricEnsemble(trained)
    asym.fit_regime_weights(dl_cal, vix_val_s, device)
    asym_f1 = asym.evaluate(dl_te, vix_test_s, device)

    # Stress Testing
    print(f"\n  [Stress Testing] ({time.time()-t_total:.1f}s)")
    stress_res = stress_test_models(trained, df_all, top_final, TARGET_COL,
                                     sc_seq, val_end.strftime('%Y-%m-%d'))

    print(f"\n  Pipeline terminé en {time.time()-t_total:.1f}s")
    return {
        'models': trained, 'calibrated': calibrated, 'stacker': stacker,
        'conformal': conformal, 'asym_ensemble': asym,
        'base_results': base_res, 'stacking_f1': stack_f1,
        'asymmetric_f1': asym_f1, 'conformal_coverage': coverage,
        'adv_auc': adv_auc, 'drift_features': drift_feats,
        'stress_results': stress_res, 'features': top_final,
        'thresholds': thresholds, 'ensemble_metrics': ens_met,
        'predictions': predictions,
    }


## 19. Exécution

In [ ]:
results = {}
for h in [5]:   # Commencer par h=5j — le plus stable dans les runs précédents
    results[h] = run_pipeline_final(df_raw, horizon=h)

# Rapport
ML_REFS = {
    'LogReg N=9 (ML h=5j GLOBAL)':       {'F1_dir':0.634,'F1_UP_FORT':0.406,'F1_DOWN_FORT':0.575},
    'RandomForest N=8 (ML h=5j GLOBAL)': {'F1_dir':0.620,'F1_UP_FORT':0.412,'F1_DOWN_FORT':0.462},
}
print("\n" + "="*70)
print("RAPPORT FINAL")
print("="*70)
for h, res in results.items():
    print(f"\n─── h={h}j ───")
    print(f"{'Modèle':<25} {'F1_dir':>8} {'F1_UP_FORT':>12} {'F1_DOWN_FORT':>14}")
    print("─"*60)
    for name, m in res['base_results'].items():
        if name == 'ENSEMBLE': continue
        print(f"  {name:<23} {m.get('F1_dir',0):>8.4f} {m.get('F1_UP_FORT',0):>12.4f} {m.get('F1_DOWN_FORT',0):>14.4f}")
    em = res['base_results'].get('ENSEMBLE',{})
    print(f"  {'ENSEMBLE':23} {em.get('F1_dir',0):>8.4f} {em.get('F1_UP_FORT',0):>12.4f} {em.get('F1_DOWN_FORT',0):>14.4f}")
    print(f"  {'STACKING':23} {res['stacking_f1']:>8.4f}")
    print(f"  {'ENSEMBLE ASYM.':23} {res['asymmetric_f1']:>8.4f}")
    print(f"  {'Conformal Coverage':23} {res['conformal_coverage']:>8.4f} (cible {1-CONFIG['conformal_alpha']:.2f})")
    print(f"  {'Adversarial AUC':23} {res['adv_auc']:>8.4f} ({'DRIFT' if res['adv_auc']>0.7 else 'OK'})")
    print("─"*60)
    for k,v in ML_REFS.items():
        print(f"  {k:<23} {v['F1_dir']:>8.4f} {v['F1_UP_FORT']:>12.4f} {v['F1_DOWN_FORT']:>14.4f}")

try:
    rows = []
    for h, res in results.items():
        for name, m in res['base_results'].items():
            rows.append({'H':h,'Modèle':name,**m})
        rows.append({'H':h,'Modèle':'Stacking','F1_dir':res['stacking_f1']})
        rows.append({'H':h,'Modèle':'Asym','F1_dir':res['asymmetric_f1']})
    for k,v in ML_REFS.items():
        rows.append({'H':5,'Modèle':k,**v})
    pd.DataFrame(rows).to_excel('vix_dl_final_report.xlsx', index=False, engine='xlsxwriter')
    print("\n[SAVE] vix_dl_final_report.xlsx")
except Exception as e:
    print(f"[WARN] Export: {e}")
print("[NOTE] Aucun modèle enregistré — validation explicite requise.")
